In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import mysql.connector
from typing import Dict

MYSQL = {
    "host"    : "localhost",    
    "port"    : 3306,
    "user"    : "root",
    "password": "Suraaj121097",  # ← replace
    "database": "global_electronics"
}

In [ ]:
DATA_DIR = Path("C:/Users/suraa/OneDrive/Documents/DataSpark")  
OUTPUT_DIR = Path("C:/Users/suraa/OneDrive/Documents/DataSpark/cleaned")  
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
import csv, os

def read_csv_flexible(path: Path, primary_encoding: str = "utf-8") -> pd.DataFrame:
    for enc in (primary_encoding, "ISO-8859-1", "latin1", "cp1252"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Unable to read {path}")

def strip_currency(series: pd.Series) -> pd.Series:
    return (series.astype(str).str.replace(r"[\$,]", "", regex=True).astype(float))

def robust_to_datetime(series: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(series, errors="coerce", dayfirst=False)
    if parsed.isna().any():
        alt = pd.to_datetime(series[parsed.isna()], errors="coerce", dayfirst=True)
        parsed.loc[parsed.isna()] = alt
    return parsed

def derive_state_code(row: pd.Series) -> str:
    raw = str(row.get("State Code", "")).strip().upper()
    name = str(row.get("State", "")).strip()
    if raw and len(raw) <= 3 and raw.isalpha():
        return raw
    abbr = "".join(w[0] for w in re.split(r"[\s\-]+", name) if w)[:3].upper()
    return abbr or None

In [ ]:
RAW = {
    "customers": read_csv_flexible(DATA_DIR / "Customers.csv"),
    "products" : read_csv_flexible(DATA_DIR / "Products.csv"),
    "sales"    : read_csv_flexible(DATA_DIR / "Sales.csv"),
    "stores"   : read_csv_flexible(DATA_DIR / "Stores.csv"),
    "exchange" : read_csv_flexible(DATA_DIR / "Exchange_Rates.csv"),
}

# Customers
customers["Birthday"] = robust_to_datetime(customers["Birthday"])
customers["State Code"] = customers.apply(derive_state_code, axis=1)

# Products – convert money strings ➜ float
products["Unit Cost USD"]  = strip_currency(products["Unit Cost USD"])
products["Unit Price USD"] = strip_currency(products["Unit Price USD"])

# Sales – rename + parse dates
sales = sales.rename(columns={"Order Date": "Order_Date", "Delivery Date": "Delivery_Date"})
sales["Order_Date"]     = pd.to_datetime(sales["Order_Date"], errors="coerce", dayfirst=False)
sales["Delivery_Date"] = pd.to_datetime(sales["Delivery_Date"], errors="coerce", dayfirst=False)

# Stores
stores["Open_Date"] = pd.to_datetime(stores["Open Date"], errors="coerce", dayfirst=False)
stores["Square Meters"] = stores["Square Meters"].fillna(stores["Square Meters"].median())

# Exchange rates
exchange["Date"] = pd.to_datetime(exchange["Date"], errors="coerce", dayfirst=False)

In [ ]:
customers.to_csv(OUTPUT_DIR / "Customers_clean.csv", index=False)
products.to_csv(OUTPUT_DIR / "Products_clean.csv", index=False)
sales.to_csv(OUTPUT_DIR / "Sales_clean.csv", index=False)
stores.to_csv(OUTPUT_DIR / "Stores_clean.csv", index=False)
exchange.to_csv(OUTPUT_DIR / "Exchange_Rates_clean.csv", index=False)

print("✅ Data cleaning & preparation complete → cleaned/*.csv")